# Module 1 - Data generation
## Medcore Pharma | Demand Planning Intelligence

Objective: Generate a realistic synthetic dataset simulating 3 years of demand planning data for 50 SKUs across 3 markets (France, Germany, Benelux).

Problem statement: S&OP exists at MedCore but KPIs are poorly interpreted.
This dataset intentionally embeds real-world pathologies that we will seek to detect:
structural forecast bias, uncaptured seasonality, bullwhip effect, business outliers and noise.

Stack: Python - pandas, numpy, SQLite

In [1]:
# Imports

import pandas as pd     #data manipulation
import numpy as np      #maths calculation
import sqlite3      #sql connection
import os   #files and folders manipulations
from datetime import datetime, timedelta #dates manipulation

# versions check
print(f"pandas : {pd.__version__}")
print(f"numpy : {np.__version__}")
print(f"sqlite3 : {sqlite3.version}")
print("All packages are ok")

pandas : 2.3.3
numpy : 2.3.5
sqlite3 : 2.6.0
All packages are ok


C:\Users\USER\AppData\Local\Temp\ipykernel_2168\1845373142.py:12: DeprecationWarning: version is deprecated and will be removed in Python 3.14
  print(f"sqlite3 : {sqlite3.version}")


In [2]:
# prject parameters

# time  period
DATE_START = datetime(2022,2,3) #first monday of 2022
DATE_END = datetime(2024,12,30)  #last monday of 2024
NB_WEEKS = 156 #3 years x 52 weeks

#markets
MARKETS = ["France", "Germany", "Benelux"]

#product families
FAMILIES = {
    "Medical Devices"       : 20,  # 20 SKUs
    "Surgical Consumables"  : 20,  # 20 SKUs
    "Diagnostic Equipment"  : 10   # 10 SKUs
}

# Average unit cost per family (in euros)
UNIT_COST = {
    "Medical Devices"       : 850,
    "Surgical Consumables"  : 45,
    "Diagnostic Equipment"  : 2200
}

# Database path
DB_PATH = os.path.join("..", "data", "medcore.db")

print(f"Period      : {DATE_START.date()} -> {DATE_END.date()}")
print(f"Weeks       : {NB_WEEKS}")
print(f"Markets     : {MARKETS}")
print(f"Families    : {list(FAMILIES.keys())}")
print(f"Database    : {DB_PATH}")
print("Parameters OK.")

Period      : 2022-02-03 -> 2024-12-30
Weeks       : 156
Markets     : ['France', 'Germany', 'Benelux']
Families    : ['Medical Devices', 'Surgical Consumables', 'Diagnostic Equipment']
Database    : ..\data\medcore.db
Parameters OK.


In [3]:
# Product Master data Generation
# We generate here the referance table for all 50 SKUs  (20+20+10) with their family, market and unit cost.
#Each SKU has a family, a  market, a unit cost a nd a pathalogy flags

import random
random.seed(42) # to be sure that we get the same results each time we run the code

skus = []
sku_id = 1

for family, nb_skus in FAMILIES.items() :
    for i in range(nb_skus) :
        market = random.choice(MARKETS)
        base_cost = UNIT_COST[family]
        unit_cost = round(base_cost *random.uniform(0.8,1.2),2)

        has_bias = random.random() <0.25 # 25% of SKUs have structural bias
        has_seasonality = random.random() < 0.30 #30% have uncaptured seasonality
        has_bullwhip = random.random() < 0.10 #10% are bullwhip amplifiers
    
        skus.append({
            "SKU_ID" :  f"SKU-{sku_id:03d}",
            "Family" : family,
            "Market" : market,
            "Unit_Cost" : unit_cost,
            "has_bias" : int(has_bias),
            "has_seasonality" : int(has_seasonality),
            "has_bullwhip" : int(has_bullwhip)
        })
        sku_id += 1

df_products = pd.DataFrame(skus)

print(f"Total SKUs generated : {len(df_products)}")
print(f"\nSKUs with bias flag        : {df_products['has_bias'].sum()}")
print(f"SKUs with seasonality flag : {df_products['has_seasonality'].sum()}")
print(f"SKUs with bullwhip flag    : {df_products['has_bullwhip'].sum()}")
print(f"\nFirst 5 rows :")
df_products.head()

Total SKUs generated : 50

SKUs with bias flag        : 20
SKUs with seasonality flag : 18
SKUs with bullwhip flag    : 7

First 5 rows :


,SKU_ID,Family,Market,Unit_Cost,has_bias,has_seasonality,has_bullwhip
0,SKU-001,Medical Devices,Benelux,717.85,0,1,0
1,SKU-002,Medical Devices,France,910.08,0,1,0
2,SKU-003,Medical Devices,France,711.86,1,0,0
3,SKU-004,Medical Devices,Benelux,900.96,0,1,0
4,SKU-005,Medical Devices,France,937.99,1,0,0


In [4]:
# Database creation & Product master data  insertion
# We create the database and insert the product master data

# Step 1 — Create the data folder if it doesn't exist
os.makedirs(os.path.join("..", "data"), exist_ok=True)

# Step 2 — Connect to the database
# If medcore.db doesn't exist yet, SQLite creates it automatically
conn = sqlite3.connect(DB_PATH)
# Step 3 — Insert the product master data into a table named "product_master"
df_products.to_sql(
    name = "product_master",
    con = conn,
    if_exists = "replace",
    index = False
)

# Step 4 — Verify that the data has been inserted correctly by querying the table
df_check = pd.read_sql_query("SELECT * FROM product_master LIMIT 5", conn)

print(f"Database created at : {DB_PATH}")
print(f"Rows written        : {len(df_products)}")
print(f"\nVerification — first 5 rows read from database :")
df_check

Database created at : ..\data\medcore.db
Rows written        : 50

Verification — first 5 rows read from database :


,SKU_ID,Family,Market,Unit_Cost,has_bias,has_seasonality,has_bullwhip
0,SKU-001,Medical Devices,Benelux,717.85,0,1,0
1,SKU-002,Medical Devices,France,910.08,0,1,0
2,SKU-003,Medical Devices,France,711.86,1,0,0
3,SKU-004,Medical Devices,Benelux,900.96,0,1,0
4,SKU-005,Medical Devices,France,937.99,1,0,0


In [5]:
# Cell 5 : Demand History Generation
# For each SKU, for each week, we generate realistic weekly demand
# with seasonalty, noise, outliers and stock on hand

# step 1 : create the date range for the demand history

weeks = pd.date_range(start=DATE_START, end=DATE_END, freq='W-MON')

print(f"Weeks generated : {len(weeks)}")
print(f"First week : {weeks[0].date()}")
print(f"Last week  : {weeks[-1].date()}")

# step 2 : Define base demand per family (units per week) without variability
BASE_DEMAND = {
    "Medical Devices"       : 15, #low volume but high value products
    "Surgical Consumables"  : 200, # high volume, low value products
    "Diagnostic Equipment"  : 10 # very low volume, very high value products
}

# step 3 - generate demand row by row

records = []

for _, sku in df_products.iterrows():
    stock = round(BASE_DEMAND[sku["Family"]]*8) #initial stock is 8 weeks of demand"
    for week in weeks:
    
          # Base demand
        base = BASE_DEMAND[sku["Family"]]

        # Seasonality layer — only for flagged SKUs
        week_number = week.isocalendar()[1]
        if sku["has_seasonality"] == 1:
            if week_number <= 8 or week_number >= 45:
                seasonality = 1.35   # +35% in winter
            elif 22 <= week_number <= 35:
                seasonality = 0.75   # -25% in summer
            else:
                seasonality = 1.0
        else:
            seasonality = 1.0

        # Noise layer — small random variation every week
        noise = np.random.normal(1.0, 0.10)  # +/- 10% random noise

        # Calculate actual demand
        actual_demand = max(0, round(base * seasonality * noise))

        # Outlier layer
        outlier_type = "none"
        if np.random.random() < 0.01:    # 1% chance of noise outlier
            actual_demand = actual_demand * round(np.random.uniform(5, 10))
            outlier_type = "noise"
        elif np.random.random() < 0.02:  # 2% chance of business outlier
            actual_demand = round(actual_demand * np.random.uniform(1.8, 2.5))
            outlier_type = "business"

        # Stock on hand calculation
        reorder = round(base * 4)        # reorder quantity = 4 weeks of supply
        if stock < round(base * 2):      # reorder point = 2 weeks of supply
            stock += reorder
        stock = max(0, stock - actual_demand)

        records.append({
            "sku_id"        : sku["SKU_ID"],
            "week"          : week.date(),
            "actual_demand" : actual_demand,
            "stock_on_hand" : stock,
            "outlier_type"  : outlier_type,
            "week_number"   : week_number
        })

# Convert to DataFrame
df_demand = pd.DataFrame(records)

print(f"Total rows generated : {len(df_demand)}")
print(f"Columns : {list(df_demand.columns)}")
print(f"\nFirst 5 rows :")
df_demand.head()

Weeks generated : 152
First week : 2022-02-07
Last week  : 2024-12-30
Total rows generated : 7600
Columns : ['sku_id', 'week', 'actual_demand', 'stock_on_hand', 'outlier_type', 'week_number']

First 5 rows :


,sku_id,week,actual_demand,stock_on_hand,outlier_type,week_number
0,SKU-001,2022-02-07,20,100,none,6
1,SKU-001,2022-02-14,18,82,none,7
2,SKU-001,2022-02-21,17,65,none,8
3,SKU-001,2022-02-28,18,47,none,9
4,SKU-001,2022-03-07,14,33,none,10


In [6]:
# Cell 6 : Forecast History generation
# We are going to generate 2 forecast sources for each sku and per week
# 1. Sales forecast - submitted by the sales team, with structural bias
# 2. Algo forecast - Simple moving average, blind to seasonality on flagged SKUs based on the current forecasting methodology

forecast_records = []

for _, sku in df_products.iterrows():
    #get the demand history for the current SKU
    sku_demand = df_demand[df_demand["sku_id"] == sku["SKU_ID"]].reset_index(drop=True)

    for i, row in sku_demand.iterrows():

        base = BASE_DEMAND[sku["Family"]]

        # Sales forecast
        # optimistic bias on flagged SKUs : Sales systematically over-forecasts
        if sku["has_bias"] == 1:
            bias_factor = np.random.uniform(1.10,1.35) #10% to 35% over-forecast
        else:
            bias_factor = np.random.uniform(0.95,1.05) #roughly accurate forecast with some noise

        sales_forecast = max(0,round(row["actual_demand"]* bias_factor))
        
        # Algo forecast
        #simple moving average of the past 4 weeks, blind to seasonality
        if i < 4:
            algo_forecast = base #not enough history, use base demand
        else:
            last_4_weeks = sku_demand.loc[i-4:i-1,"actual_demand"]
            algo_forecast = max(0, round(last_4_weeks.mean()))

        # Forecast source flag
        # # In real life, the final forecast is sometimes Sales, sometimes Algo
        # Here we simulate that Sales overrides Algo 60% of the time
        if np.random.random() < 0.60:
            final_forecast = sales_forecast
            forecast_source = "Sales"
        else:
            final_forecast = algo_forecast
            forecast_source = "Algo"

        forecast_records.append({
            "sku_id"          : sku["SKU_ID"],
            "week"            : row["week"],
            "sales_forecast"  : sales_forecast,
            "algo_forecast"   : algo_forecast,
            "final_forecast"  : final_forecast,
            "forecast_source" : forecast_source
        })

# Convert to DataFrame
df_forecast = pd.DataFrame(forecast_records)

print(f"Total rows generated : {len(df_forecast)}")
print(f"Columns : {list(df_forecast.columns)}")
print(f"\nForecast source split :")
print(df_forecast["forecast_source"].value_counts())
print(f"\nFirst 5 rows :")
df_forecast.head()

        

            

Total rows generated : 7600
Columns : ['sku_id', 'week', 'sales_forecast', 'algo_forecast', 'final_forecast', 'forecast_source']

Forecast source split :
forecast_source
Sales    4547
Algo     3053
Name: count, dtype: int64

First 5 rows :


,sku_id,week,sales_forecast,algo_forecast,final_forecast,forecast_source
0,SKU-001,2022-02-07,21,15,21,Sales
1,SKU-001,2022-02-14,19,15,19,Sales
2,SKU-001,2022-02-21,17,15,17,Sales
3,SKU-001,2022-02-28,17,15,15,Algo
4,SKU-001,2022-03-07,14,18,18,Algo


In [7]:
# Cell 7 : Save all tables to sqLite database
# We write df_demand and df_forecast to the database and verify all 3 tables are present and correct 

#saving demand history to database
df_demand.to_sql(
    name = "demand_history",
    con = conn,
    if_exists = "replace",
    index = False
)

# Save forecast history
df_forecast.to_sql(
    name      = "forecast_history",
    con       = conn,
    if_exists = "replace",
    index     = False
)

#verification step
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type ='table'",conn)
print(f"Tables in database : {list(tables['name'])}")    

# row count per table
for table in tables['name']:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn).iloc[0]['n']
    print(f" {table} : {count} rows")

# closse de connection
conn.close()
print("\nDatabase connection closed")
print("Module 1 complete.")

Tables in database : ['product_master', 'demand_history', 'forecast_history']
 product_master : 50 rows
 demand_history : 7600 rows
 forecast_history : 7600 rows

Database connection closed
Module 1 complete.
